# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ibrahimcancode/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of Analysis

For this project, one row represents one webpage (content item) for one client during a selected time window. The analysis will use a mid-panel month (March 2026) from the warehouse to avoid using future information.

The goal is to group webpages with similar SEO and content characteristics into meaningful content archetypes. This allows SEO teams to identify different types of pages and decide what optimization strategy is most appropriate for each group.

In [5]:
import duckdb

con = duckdb.connect()

In [9]:
import os, getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN is None:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")



import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


Paste your Hugging Face READ token (hf_...): ··········
dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [13]:
con.sql("SHOW TABLES").df()

,name
0,dim_clients
1,dim_content
2,fact_daily
3,fact_daily_sample
4,fact_query_90d


In [12]:
for name, src in TABLES.items():
    con.sql(f"""
        CREATE OR REPLACE VIEW {name} AS
        SELECT * FROM {src}
    """)

In [14]:
con.sql("DESCRIBE fact_daily").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip -q install duckdb huggingface_hub

In [8]:
con.sql("SHOW TABLES").df()

,name


In [15]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id) AS total_clients,
    COUNT(DISTINCT content_hash_id) AS total_pages,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM fact_daily
WHERE month = '2026-03'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,total_clients,total_pages,first_date,last_date
0,9841378,55,331437,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Fields

### Features
- Impressions
- Clicks
- Average Position
- CTR
- Word Count

These describe webpage performance and content characteristics before any decision is made.

### Label / Proxy

Since clustering is an unsupervised learning task, there is no predefined label. The clusters themselves act as a proxy for different content archetypes.

### Context

- Client Hash ID
- Content Hash ID
- Report Date

These provide context for each observation but are not used to define the clusters.

### Excluded

I excluded future performance metrics and any information that would reveal outcomes after the decision point. This prevents data leakage and keeps the analysis realistic.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    ga4_engaged_sessions
FROM fact_daily
WHERE month='2026-03'
LIMIT 10
""").df()

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,<NA>,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,<NA>,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,<NA>,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,<NA>,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,<NA>,<NA>
5,2026-03-01,client_73cda7b4e4f265ea,content_36c36abc7650d7af,239,1,7.347280,<NA>,<NA>
6,2026-03-01,client_73cda7b4e4f265ea,content_a7da352b73b02668,191,0,7.832461,<NA>,<NA>
7,2026-03-01,client_73cda7b4e4f265ea,content_05434271b257bb68,55,0,3.272727,<NA>,<NA>
8,2026-03-01,client_73cda7b4e4f265ea,content_d056587ff7faca0c,77,0,5.636364,<NA>,<NA>
9,2026-03-01,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2,0,4.500000,<NA>,<NA>


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## Data Verification

The following queries verify the grain of the data, the amount of available data, and the availability of Google Search Console information for the selected month.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Query 1: Grain")

display(con.sql("""
SELECT
COUNT(*) AS rows,
COUNT(DISTINCT client_hash_id) AS clients,
COUNT(DISTINCT content_hash_id) AS pages
FROM fact_daily
WHERE month='2026-03'
""").df())

print("Query 2: Date Window")

display(con.sql("""
SELECT
MIN(report_date) AS first_date,
MAX(report_date) AS last_date
FROM fact_daily
WHERE month='2026-03'
""").df())

print("Query 3: Available GSC Data")

display(con.sql("""
SELECT
COUNT(*) AS available_rows
FROM fact_daily
WHERE month='2026-03'
AND gsc_data_available IS TRUE
""").df())

Query 1: Grain


,rows,clients,pages
0,9841378,55,331437


Query 2: Date Window


,first_date,last_date
0,2026-03-01,2026-03-31


Query 3: Available GSC Data


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,3611061


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data Limits

This dataset contains observed search and analytics data only. It cannot explain why Google ranks webpages in a certain way or prove cause-and-effect relationships.

The clusters discovered by this project should be treated as decision-support insights rather than predictions of Google's algorithm.

The analysis is also limited to the available historical data and the selected time window.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql("""
SELECT
COUNT(*) AS total_rows,
COUNT(CASE WHEN ga4_data_available IS FALSE THEN 1 END) AS rows_without_ga4,
COUNT(CASE WHEN gsc_data_available IS FALSE THEN 1 END) AS rows_without_gsc
FROM fact_daily
WHERE month='2026-03'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,rows_without_ga4,rows_without_gsc
0,9841378,6408671,6230317


## Data Limits

This dataset cannot explain why Google ranks webpages in a certain way or prove cause-and-effect relationships. It only contains observed search-performance data.

The analysis also depends on the available historical records, so some webpages may have shorter histories than others. Any discovered archetypes should therefore be interpreted as decision-support rather than proof of ranking behavior.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.